# **ML Models for an accurate GSE prediction**

## **Step 1. Import dataset**

In [ ]:
import pandas as pd
import numpy as np


data = pd.read_csv('RFE_SolCbio3_descs_fps.csv') #import filtered descriptors
data.drop(columns=['smiles'], inplace=True) #remove unwanted columns

## **Step 2. Stratified sampling**
We want a balanced training and test set.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Create a new column 'output' by mapping 'dev' column
data['output'] = data['dev'].map({'NO': 0, 'YES': 1})

# Delete the original 'dev' column
data.drop(columns=['dev'], inplace=True)


X = data.drop('output', axis=1)
y = data['output']


sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in sss.split(X, y):
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Now you have stratified samples in X_train, X_test, y_train, and y_test

print(f'Training set (80%): {X_train.shape}')
print(f'Test set (20%): {X_test.shape}')
print(f'Accurate predictions of GSE in test set: {100*round(y_test.value_counts()[0]/len(y_test),3)} %')
print(f'Accurate predictions of GSE in training set: {100*round(y_train.value_counts()[0]/len(y_train),3)} %')

## **Step 3. Random Forest**
Our first classification model will be a Random Forest.

1. Train te RF with the training set.
2. Optimize hyperparameters.
3. Predict the test set.
3. Report Accuracy, sensitivity, and specificity.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RandomizedSearchCV

# hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}


# Instantiate a RandomForestClassifier object
RF_rand = RandomForestClassifier(random_state=42)

# Instantiate a RandomizedSearchCV object
# n_iter controls the number of random combinations to try
random_search = RandomizedSearchCV(estimator=RF_rand, param_distributions=param_grid, n_iter=10, cv=10, scoring='accuracy', n_jobs=-1, random_state=42)

# Fit the RandomizedSearchCV object to the training data
random_search.fit(X_train, y_train)

# Print the best hyperparameters
print("Best hyperparameters found by RandomizedSearchCV:")
print(random_search.best_params_)

In [ ]:
RF_optimized = RandomForestClassifier(random_state=42, **random_search.best_params_) #train RF with tuned parameters
RF_optimized.fit(X_train, y_train)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

# Predict on the test set
y_pred_RF = RF_optimized.predict(X_test)

# Calculate the confusion matrix
cm = confusion_matrix(y_test, y_pred_RF)

# Calculate accuracy, sensitivity, and specificity
tp, fn, fp, tn = cm.ravel()

rf_accuracy = (tp + tn) / (tp + tn + fp + fn)
rf_sensitivity = tp / (tp + fn)
rf_specificity = tn / (tn + fp)

# Print the results
print(f'Optimized Model Accuracy: {rf_accuracy:.4f}')
print(f'Optimized Model Sensitivity: {rf_sensitivity:.4f}')
print(f'Optimized Model Specificity: {rf_specificity:.4f}')


# Plot the confusion matrix
plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Accurate GSE', 'Unaccurate GSE'], yticklabels=['Accurate GSE', 'Unaccurate GSE'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

RF_ac = round((cm[0,0]+cm[1,1])/(cm[0,0]+cm[1,1]+cm[0,1]+cm[1,0]),2)

print(f'The accuracy of the Random Forest is {RF_ac}')

## **Step 4: Support Vector Machine**


1. Train SVM with the training set. Try every kernel.
2. Optimize hyperparameters.
3. Predict the test set.
3. Report Accuracy, sensitivity, and specificity.

In [ ]:
from sklearn.svm import SVC

svm_kernels = [ 'poly', 'rbf', 'sigmoid']

param_grids = {
    #'linear': {'C': [0.1, 1, 10, 100]},
    'poly': {'C': [0.1, 1, 10, 100], 'degree': [2, 3, 4], 'gamma': ['scale', 'auto'], 'coef0': [0.0, 0.1, 0.5]},
    'rbf': {'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto']},
    'sigmoid': {'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto'], 'coef0': [0.0, 0.1, 0.5]}
}

In [ ]:
best_params_per_kernel = {}
best_estimators_per_kernel = {}

for i, kernel in enumerate(svm_kernels):
    print(f"Running RandomizedSearchCV for {kernel} kernel (Iteration {i+1}/{len(svm_kernels)})...")
    svm_model = SVC(kernel=kernel, random_state=42)
    random_search_svm = RandomizedSearchCV(estimator=svm_model, param_distributions=param_grids[kernel], n_iter=10, cv=3, scoring='accuracy', n_jobs=-1, random_state=42) # Reduced cv to 3 for faster testing
    random_search_svm.fit(X_train, y_train)
    best_params_per_kernel[kernel] = random_search_svm.best_params_
    best_estimators_per_kernel[kernel] = random_search_svm.best_estimator_
    print(f"Best hyperparameters for {kernel} kernel: {random_search_svm.best_params_}")

print("\nBest parameters per kernel:")
print(best_params_per_kernel)

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

evaluation_results = {}

for kernel, best_estimator in best_estimators_per_kernel.items():
    print(f"Evaluating {kernel} kernel...")

    # Predict on the test set
    y_pred_svm = best_estimator.predict(X_test)

    # Calculate the confusion matrix
    cm_svm = confusion_matrix(y_test, y_pred_svm)

    # Calculate accuracy, sensitivity, and specificity
    # Handle the case where a class might not be predicted
    tp_svm, fn_svm, fp_svm, tn_svm = cm_svm.ravel() if cm_svm.size == 4 else (0, 0, 0, 0) # Initialize if confusion matrix is not 2x2

    accuracy = (tp_svm + tn_svm) / (tp_svm + tn_svm + fp_svm + fn_svm) if (tp_svm + tn_svm + fp_svm + fn_svm) > 0 else 0
    sensitivity = tp_svm / (tp_svm + fn_svm) if (tp_svm + fn_svm) > 0 else 0
    specificity = tn_svm / (tn_svm + fp_svm) if (tn_svm + fp_svm) > 0 else 0


    evaluation_results[kernel] = {
        'accuracy': accuracy,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'confusion_matrix': cm_svm
    }

    # Print the results
    print(f'{kernel} Kernel Accuracy: {accuracy:.4f}')
    print(f'{kernel} Kernel Sensitivity: {sensitivity:.4f}')
    print(f'{kernel} Kernel Specificity: {specificity:.4f}')

    # Plot the confusion matrix
    plt.figure(figsize=(4, 4))
    sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Accurate GSE', 'Unaccurate GSE'], yticklabels=['Accurate GSE', 'Unaccurate GSE'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix - {kernel} Kernel')
    plt.show()

## **Step 5: XGBoost Classifier**

1. Train XGBoost with the training set.
2. Optimize hyperparameters.
3. Predict the test set.
4. Report Accuracy, sensitivity, and specificity.

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier

# Instantiate XGBoost Classifier
xgb_model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss') # Set eval_metric to avoid warning

# Define the hyperparameter grid for RandomizedSearchCV
param_grid_xgb = {
    'n_estimators': [100, 200, 300, 400, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'max_depth': [3, 5, 7, 9, 11],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.001, 0.005, 0.01, 0.05, 0.1],
    'reg_lambda': [0, 0.001, 0.005, 0.01, 0.05, 0.1]
}

# Instantiate RandomizedSearchCV for XGBoost
random_search_xgb = RandomizedSearchCV(estimator=xgb_model, param_distributions=param_grid_xgb, n_iter=10, cv=3, scoring='accuracy', n_jobs=-1, random_state=42)

# Fit the RandomizedSearchCV object to the training data
random_search_xgb.fit(X_train, y_train)

# Print the best hyperparameters
print("Best hyperparameters found by RandomizedSearchCV for XGBoost:")
print(random_search_xgb.best_params_)

# Train the XGBoost model with the best hyperparameters
xgb_optimized = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', **random_search_xgb.best_params_)
xgb_optimized.fit(X_train, y_train)

# Predict on the test set
y_pred_xgb = xgb_optimized.predict(X_test)

# Calculate the confusion matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

# Calculate accuracy, sensitivity, and specificity
tp_xgb, fn_xgb, fp_xgb, tn_xgb = cm_xgb.ravel()

accuracy_xgb = (tp_xgb + tn_xgb) / (tp_xgb + tn_xgb + fp_xgb + fn_xgb)
sensitivity_xgb = tp_xgb / (tp_xgb + fn_xgb)
specificity_xgb = tn_xgb / (tn_xgb + fp_xgb)

# Print the results
print(f'Optimized XGBoost Model Accuracy: {accuracy_xgb:.4f}')
print(f'Optimized XGBoost Model Sensitivity: {sensitivity_xgb:.4f}')
print(f'Optimized XGBoost Model Specificity: {specificity_xgb:.4f}')

# Plot the confusion matrix
plt.figure(figsize=(4, 4))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Accurate GSE', 'Unaccurate GSE'], yticklabels=['Accurate GSE', 'Unaccurate GSE'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - XGBoost')
plt.show()

## **Step 6: Naive Bayes Classifier**

1. Train Naive Bayes with the training set.
2. Optimize hyperparameters.
3. Predict the test set.
4. Report Accuracy, sensitivity, and specificity.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Instantiate Gaussian Naive Bayes Classifier
nb_model = GaussianNB()

# Define the hyperparameter grid for GridSearchCV
# GaussianNB doesn't have many hyperparameters to tune.
# We can tune 'var_smoothing', which is a small value added to the variances for calculation stability.
param_grid_nb = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}

# Instantiate GridSearchCV for Naive Bayes
grid_search_nb = GridSearchCV(estimator=nb_model, param_grid=param_grid_nb, cv=5, scoring='accuracy', n_jobs=-1)

# Fit the GridSearchCV object to the training data
grid_search_nb.fit(X_train, y_train)

# Print the best hyperparameters
print("Best hyperparameters found by GridSearchCV for Naive Bayes:")
print(grid_search_nb.best_params_)

# Train the Naive Bayes model with the best hyperparameters
nb_optimized = GaussianNB(**grid_search_nb.best_params_)
nb_optimized.fit(X_train, y_train)

# Predict on the test set
y_pred_nb = nb_optimized.predict(X_test)

# Calculate the confusion matrix
cm_nb = confusion_matrix(y_test, y_pred_nb)

# Calculate accuracy, sensitivity, and specificity
tp_nb, fn_nb, fp_nb, tn_nb = cm_nb.ravel()

accuracy_nb = (tp_nb + tn_nb) / (tp_nb + tn_nb + fp_nb + fn_nb)
sensitivity_nb = tp_nb / (tp_nb + fn_nb)
specificity_nb = tn_nb / (tn_nb + fp_nb)

# Print the results
print(f'Optimized Naive Bayes Model Accuracy: {accuracy_nb:.4f}')
print(f'Optimized Naive Bayes Model Sensitivity: {sensitivity_nb:.4f}')
print(f'Optimized Naive Bayes Model Specificity: {specificity_nb:.4f}')

# Plot the confusion matrix
plt.figure(figsize=(4, 4))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Accurate GSE', 'Unaccurate GSE'], yticklabels=['Accurate GSE', 'Unaccurate GSE'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Naive Bayes')
plt.show()

## **Step 7: Artificial Neural Network**

In [ ]:
!pip install tensorflow
!pip install scikeras

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from scikeras.wrappers import KerasClassifier


# Calculate class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

# Improved model
NN = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(32, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

# Optimizer with custom learning rate
optimizer = Adam(learning_rate=0.0005)

NN.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Callbacks
early_stop = EarlyStopping(monitor='accuracy', patience=100,
                          restore_best_weights=True, mode='max')
reduce_lr = ReduceLROnPlateau(monitor='accuracy', factor=0.5,
                             patience=20, min_lr=1e-7)


kcv_NN = KerasClassifier(
    model=NN,
    epochs=1000,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weight_dict,
    verbose=0
  )

# Fit the KerasClassifier model to the training data
kcv_NN.fit(X_train, y_train)

# Evaluate
y_pred_NN_binary = (kcv_NN.predict(X_test) > 0.5).astype(int)

In [ ]:
# Calculate the confusion matrix
cm_NN = confusion_matrix(y_test, y_pred_NN_binary)

# Calculate accuracy, sensitivity, and specificity
tp_NN, fn_NN, fp_NN, tn_NN = cm_NN.ravel()
accuracy_NN = (tp_NN + tn_NN) / (tp_NN + tn_NN + fp_NN + fn_NN)
sensitivity_NN = tp_NN / (tp_NN + fn_NN)
specificity_NN = tn_NN / (tn_NN + fp_NN)

# Print the results
print(f'Optimized Neural Network Model Accuracy: {accuracy_NN:.4f}')
print(f'Optimized Neural Network Model Sensitivity: {sensitivity_NN:.4f}')
print(f'Optimized Neural Network Model Specificity: {specificity_NN:.4f}')

# Plot the confusion matrix
plt.figure(figsize=(4, 4))
sns.heatmap(cm_NN, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Accurate GSE', 'Unaccurate GSE'], yticklabels=['Accurate GSE', 'Unaccurate GSE'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Neural Network')
plt.show()

In [ ]:
NN.save('nn_model.keras')
print("Artificial Neural Network model saved as 'nn_model.keras'")

**Average of best models**

In [ ]:
y_pred_avg = (y_pred_RF*0.5 + y_pred_NN_binary*0.5).astype(int)
cm_avg = confusion_matrix(y_test,y_pred_avg)
tp_avg, fn_avg, fp_avg, tn_avg = cm_avg.ravel()
avg_accuracy = (tp_avg + tn_avg) / (tp_avg + tn_avg + fp_avg + fn_avg)
avg_sensitivity = tp_avg / (tp_avg + fn_avg)
avg_specificity = tn_avg / (tn_avg + fp_avg)

# Print the results
print(f'Average NN + RF Accuracy: {avg_accuracy:.4f}')
print(f'Average NN + RF Sensitivity: {avg_sensitivity:.4f}')
print(f'Average NN + RF Specificity: {avg_specificity:.4f}')

plt.figure(figsize=(4, 4))
sns.heatmap(cm_avg, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Accurate GSE', 'Unaccurate GSE'], yticklabels=['Accurate GSE', 'Unaccurate GSE'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Neural Network')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Combine the metrics into a single dictionary for easier processing
combined_metrics = {
    'Random Forest': {
        'Accuracy': rf_accuracy,
        'Sensitivity': rf_sensitivity,
        'Specificity': rf_specificity
    },
    'SVM (RBF Kernel)': {
        'Accuracy': evaluation_results['rbf']['accuracy'],
        'Sensitivity': evaluation_results['rbf']['sensitivity'],
        'Specificity': evaluation_results['rbf']['specificity']
    },
    'XGBoost': {
        'Accuracy': accuracy_xgb,
        'Sensitivity': sensitivity_xgb,
        'Specificity': specificity_xgb
    },
    'Neural Network': {
        'Accuracy': accuracy_NN,
        'Sensitivity': sensitivity_NN, # Corrected
        'Specificity': specificity_NN  # Corrected
    },
    'Average': {
        'Accuracy': avg_accuracy,
        'Sensitivity': avg_sensitivity,
        'Specificity': avg_specificity
    }
}

# Convert to DataFrame
df_metrics = pd.DataFrame.from_dict(combined_metrics, orient='index')
df_metrics.index.name = 'Model'

# Melt the DataFrame to long format
df_melted = df_metrics.reset_index().melt(id_vars='Model', var_name='Metric', value_name='Score')

# Plotting
plt.figure(figsize=(7, 6))
ax = sns.barplot(y='Model', x='Score', hue='Metric', data=df_melted, palette=['#322982', '#5d5d5c', '#a1a0a1'])
ax.set_xlabel('Score', fontsize=16)
ax.set_ylabel('', fontsize=16)
ax.set_xlim(0, 1.05)
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)
ax.legend(title='', fontsize=12, loc='upper center', ncols = 3,bbox_to_anchor=(0,1.01,1,0.08))
plt.grid(True, axis='x', linestyle='--', color='gray', alpha=0.7)

# Add values inside the bars
for p in ax.patches:
    width = p.get_width()
    # Place text just inside the bar, aligned to the right
    plt.text(width - 0.02, # X-coordinate for the text, slightly inside the bar
             p.get_y() + p.get_height() / 2, # Y-coordinate for the text
             f'{width:.2f}', # Value to display
             ha='right', # Horizontal alignment to the right
             va='center', # Vertical alignment to the center
             color='white') # Text color for better visibility

plt.tight_layout()
plt.savefig('performance_ML.png', format='png', dpi=300, bbox_inches='tight')
plt.savefig('performance_ML.pdf', format='pdf', dpi=500, bbox_inches='tight')
plt.show()

## **Step 8: *k*-fold cross validation**

In [ ]:
from sklearn.model_selection import cross_val_score


scores_rf = cross_val_score(RF_optimized, X, y, cv = 10)
scores_svm = cross_val_score(best_estimators_per_kernel['rbf'], X, y, cv = 10)
scores_xgb = cross_val_score(xgb_optimized, X, y, cv = 10)


from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler # Import StandardScaler for NN cross-validation
import numpy as np # Ensure numpy is available for class_weight

# Manual cross-validation
def create_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)), # Input dimension needs to be dynamic
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(32, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    optimizer = Adam(learning_rate=0.0005)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Perform manual K-fold cross validation
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores_NN = []

# Initialize a scaler for the Neural Network, as it expects scaled data.
# The scaler will be fitted for each fold to avoid data leakage.
scaler_nn_cv = StandardScaler()

for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y)):
    print(f"Fold {fold + 1}/10")

    # Split data using .iloc for correct row selection
    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    # Scale data for the Neural Network within each fold
    X_train_fold_scaled = scaler_nn_cv.fit_transform(X_train_fold)
    X_val_fold_scaled = scaler_nn_cv.transform(X_val_fold)

    # Calculate class weights for this fold
    class_weights_fold = compute_class_weight('balanced',
                                            classes=np.unique(y_train_fold),
                                            y=y_train_fold)
    class_weight_dict_fold = {0: class_weights_fold[0], 1: class_weights_fold[1]}

    # Create and train model
    model = create_model(X_train_fold_scaled.shape[1]) # Pass input_dim

    history = model.fit(
        X_train_fold_scaled, y_train_fold,
        epochs=1000,
        batch_size=32,
        callbacks=[early_stop, reduce_lr],
        class_weight=class_weight_dict_fold,
        validation_data=(X_val_fold_scaled, y_val_fold),
        verbose=0
    )

    # Evaluate
    y_pred = (model.predict(X_val_fold_scaled) > 0.5).astype(int)
    accuracy = accuracy_score(y_val_fold, y_pred)
    scores_NN.append(accuracy)
    print(f"NN Fold {fold + 1} Accuracy: {accuracy:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a DataFrame from the cross-validation scores
cv_results = pd.DataFrame({
    'Random Forest': scores_rf,
    'Support Vector Machine': scores_svm,
    'XGBoost': scores_xgb,
    'Neural Network': scores_NN
})

# Melt the DataFrame to a long format for seaborn plotting
cv_results_melted = cv_results.melt(var_name='Model', value_name='Accuracy')

# Define the order for the x-axis
model_order = ["Neural Network", "Support Vector Machine", "Random Forest", "XGBoost"]

# Create the figure and axes for the plot
plt.figure(figsize=(7, 6))
ax = plt.gca()

# Create the violin plot
sns.violinplot(y='Model', x='Accuracy', data=cv_results_melted, inner=None, palette=['#322982', '#625f92', '#5d5d5c', '#a1a0a1'], order=model_order, ax=ax)

# Overlay the jitter plot (stripplot) for individual data points
sns.stripplot(y='Model', x='Accuracy', data=cv_results_melted, jitter=True, color='black', size=4, order=model_order, ax=ax)

# Calculate and add average accuracy and standard deviation as text boxes with rounded borders
for i, model_name in enumerate(model_order):
    model_scores = cv_results_melted[cv_results_melted['Model'] == model_name]['Accuracy']
    mean_accuracy = model_scores.mean()
    std_accuracy = model_scores.std()

    # Position the text slightly to the right of the plot origin
    # Use ax.annotate for text boxes with styling
    ax.annotate(f'Mean: {mean_accuracy:.3f}',
                xy=(0.95, i + 0.1),
                xycoords='data',
                ha='right', va='center', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', fc='lightgray', ec='black', lw=0.5))
    ax.annotate(f'Std: {std_accuracy:.3f}',
                xy=(0.95, i - 0.1),
                xycoords='data',
                ha='right', va='center', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', fc='lightgray', ec='black', lw=0.5))

ax.set_xlabel('Accuracy Score', fontsize=16)
ax.set_ylabel('', fontsize=16)
ax.set_xlim(0.4, 1.0) # Adjusted x-axis limit for text boxes
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('cv_accuracy_distribution.png', format='png', dpi=300, bbox_inches='tight')
plt.savefig('cv_accuracy_distribution.pdf', format='pdf', dpi=500, bbox_inches='tight')
plt.show()